# Traffic → Bridge-Level Feature Engineering — FINAL WORKFLOW

### Unit-safe version

Traffic DTV values are vehicle counts per day (Kfz/d). The raw BASt files may use a period as a thousands separator (for example `2.312` = 2,312 vehicles/day). Therefore DTV is parsed from the original text before feature engineering. Heavy-vehicle share is then calculated as `heavy_vehicle_volume / dtv`, using the same vehicle-count unit.


# DATA TRANSFER / INPUT–OUTPUT MANIFEST — Notebook 02

## Portable location rule
All local paths are derived from one `PROJECT_ROOT`. On another computer:
1. put `Dataset_PlanA-B` under the project root and start Jupyter from that root, **or**
2. set `BRIDGE_PROJECT_ROOT` to the project root.

Required local Traffic structure:

```text
<PROJECT_ROOT>/
├── Dataset_PlanA-B/
│   └── Traffic/
│       ├── raw/
│       ├── traffic_raw_combined.csv
│       └── traffic_raw_combined.parquet
└── Output_PlanA-B/
```

## Data inventory

| Class | Data | How obtained/read | Role |
|---|---|---|---|
| External API | Mobilithek metadata | `METADATA_API_URL` via `requests.get()` | Defines available Traffic resources |
| External download | Annual BASt Traffic CSVs | URLs returned by Mobilithek API | Raw source data |
| Local raw cache | Annual Traffic CSVs | `Dataset_PlanA-B/Traffic/raw/` | Reusable downloaded source |
| Local cache | `traffic_raw_combined.parquet` | `pd.read_parquet()` on normal runs | Main local Traffic cache |
| Local audit artifact | `traffic_raw_combined.csv` | Written during rebuild | Audit/reuse; not the canonical downstream interface |
| PostgreSQL input | `final.bridge` and required Traffic station layers | SQL | Bridge/station integration |
| PostgreSQL output | raw/cleaned/transformed Traffic layers | SQL | Processing layers |
| PostgreSQL output | `final.traffic` | SQL | **Canonical downstream Traffic output** |

## Downstream rule

Notebook 04 consumes **`final.traffic`**. It does not need to read the annual Traffic CSV files or the combined Traffic Parquet directly.

The local files are therefore source/cache/intermediate artifacts. `final.traffic` is the canonical interface to the next notebook.

## Transfer to another computer

For offline reproducibility, transfer the complete `Dataset_PlanA-B/Traffic/` directory and the PostgreSQL database/export. With internet access, the Traffic raw cache can be rebuilt from the documented Mobilithek API/resource URLs.


## 1. Setup & Configuration

One configuration block. The existing project table names and the 5 km spatial matching rule are kept.

In [1]:

from getpass import getpass
import re
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

PG_HOST = "localhost"
PG_PORT = 5432
PG_DATABASE = "Final_Project"
PG_USER = "postgres"
PG_PASSWORD = getpass("Enter PostgreSQL password: ")

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}"
    f"@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
)

with engine.connect() as conn:
    print("PostgreSQL connection:", conn.execute(
        text("SELECT current_database();")
    ).scalar())


import os
from pathlib import Path

def find_project_root():
    env_root = os.environ.get("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT is set, but Dataset_PlanA-B was not found under: {root}"
        )

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Start Jupyter from the project root "
        "or set BRIDGE_PROJECT_ROOT."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output root:", OUTPUT_ROOT)

# ---------------- Traffic local artifacts ----------------
TRAFFIC_DIR = DATASET_ROOT / "Traffic"
RAW_TRAFFIC_DIR = TRAFFIC_DIR / "raw"
COMBINED_TRAFFIC_FILE = TRAFFIC_DIR / "traffic_raw_combined.csv"
TRAFFIC_PARQUET_FILE = TRAFFIC_DIR / "traffic_raw_combined.parquet"

RAW_TRAFFIC_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- External Traffic source ----------------
DATASET_ID = "573280210227470336"
METADATA_API_URL = (
    f"https://mobilithek.info/mdp-api/mdp-msa-metadata/v2/"
    f"offers/{DATASET_ID}"
)

EXCLUDED_RESOURCES = {18}
MIN_BUILD_YEAR = 1900
MAX_TRAFFIC_MATCH_DISTANCE_KM = 5.0

print("Traffic directory:", TRAFFIC_DIR)
print("Traffic raw cache:", RAW_TRAFFIC_DIR)
print("Traffic Parquet:", TRAFFIC_PARQUET_FILE)


PostgreSQL connection: Final_Project
Project root: C:\Datenanalyse\final Project
Dataset root: C:\Datenanalyse\final Project\Dataset_PlanA-B
Output root: C:\Datenanalyse\final Project\Output_PlanA-B
Traffic directory: C:\Datenanalyse\final Project\Dataset_PlanA-B\Traffic
Traffic raw cache: C:\Datenanalyse\final Project\Dataset_PlanA-B\Traffic\raw
Traffic Parquet: C:\Datenanalyse\final Project\Dataset_PlanA-B\Traffic\traffic_raw_combined.parquet


## 2. Source & Raw Traffic Layer

Preserve the original annual BASt traffic distributions and synchronize the combined raw data with PostgreSQL.

## 2.1 Source Metadata & Resource Selection

Retrieve the official Mobilithek metadata and define the annual CSV resources used by the project.
Resource 18 remains excluded because it is not a valid annual CSV distribution.

In [2]:
response = requests.get(
    METADATA_API_URL,
    timeout=30,
    headers={"Accept": "application/json"}
)
response.raise_for_status()
metadata = response.json()

traffic_metadata_df = pd.DataFrame([{
    "dataset_id": metadata.get("id"),
    "publication_id": metadata.get("publicationId"),
    "title": metadata.get("title"),
    "description": metadata.get("description"),
    "temporal_coverage": str(metadata.get("temporalCoverage")),
    "access_rights": metadata.get("accessRights"),
    "language": metadata.get("language"),
    "created": metadata.get("created"),
    "modified": metadata.get("modified")
}])

traffic_resources_df = pd.DataFrame([
    {
        "dataset_id": DATASET_ID,
        "resource_number": i,
        "access_url": r.get("accessUrl"),
        "access_protocol": r.get("accessProtocol"),
        "format_description": r.get("dataFormatAdditionalDescription")
    }
    for i, r in enumerate(metadata.get("contentData", []), start=1)
])

valid_resources_df = traffic_resources_df[
    ~traffic_resources_df["resource_number"].isin(EXCLUDED_RESOURCES)
].copy()

valid_resources_df = valid_resources_df[
    valid_resources_df["access_url"].notna()
    & valid_resources_df["access_url"].str.contains(
        ".csv", case=False, regex=False
    )
].copy()

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS metadata"))

traffic_metadata_df.to_sql(
    "traffic_dataset", engine, schema="metadata",
    if_exists="replace", index=False
)
traffic_resources_df.to_sql(
    "traffic_resources", engine, schema="metadata",
    if_exists="replace", index=False
)

print("Dataset:", traffic_metadata_df.loc[0, "title"])
print("Resources selected:", len(valid_resources_df))
print("Excluded:", sorted(EXCLUDED_RESOURCES))
display(valid_resources_df[["resource_number", "access_url"]])

Dataset: Automatische Zählstellen auf Autobahnen und Bundesstraßen
Resources selected: 22
Excluded: [18]


,resource_number,access_url
0,1,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
1,2,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
2,3,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
3,4,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
4,5,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
5,6,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
6,7,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
7,8,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
8,9,https://www.bast.de/DE/Themen/Digitales/HF_1/M...
9,10,https://www.bast.de/DE/Themen/Digitales/HF_1/M...


## 2.2 Raw Acquisition & Parquet Cache

The project uses one canonical Traffic Parquet dataset at the Dataset_PlanA-B root. If it already exists, it is loaded directly and the annual CSV files are not re-read. Set `FORCE_REBUILD_TRAFFIC_PARQUET = True` only when the source Traffic data changes.


In [3]:
def detect_csv(file_path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    separators = [",", ";", "\t"]
    for enc in encodings:
        for sep in separators:
            try:
                sample = pd.read_csv(file_path, encoding=enc, sep=sep, nrows=5,
                                     dtype=str, keep_default_na=False)
                if len(sample.columns) > 1:
                    return enc, sep
            except Exception:
                pass
    return None, None

# Normal run: reuse the existing Parquet dataset.
# Set to True only when the source Traffic data has changed and the Parquet must be rebuilt.
FORCE_REBUILD_TRAFFIC_PARQUET = False

if TRAFFIC_PARQUET_FILE.exists() and not FORCE_REBUILD_TRAFFIC_PARQUET:
    traffic_raw_combined = pd.read_parquet(TRAFFIC_PARQUET_FILE, engine="pyarrow")
    print("Existing Traffic Parquet loaded:", TRAFFIC_PARQUET_FILE)
    print("Rows:", f"{len(traffic_raw_combined):,}")
    print("Columns:", len(traffic_raw_combined.columns))
    print("Resources:", traffic_raw_combined["resource_number"].nunique())
    print("Years:", sorted(traffic_raw_combined["traffic_year"].dropna().unique().tolist()))
else:
    download_log = []

    for _, row in valid_resources_df.sort_values("resource_number").iterrows():
        resource_number = int(row["resource_number"])
        output_path = RAW_TRAFFIC_DIR / f"traffic_resource_{resource_number}.csv"

        if output_path.exists() and output_path.stat().st_size > 0:
            status = "existing"
        else:
            r = requests.get(row["access_url"], timeout=120,
                             headers={"User-Agent": "Mozilla/5.0"})
            r.raise_for_status()
            content = r.content
            head = content[:100].lstrip().lower()
            if head.startswith(b"<!doctype html") or b"<html" in head[:100]:
                raise ValueError("Server returned HTML instead of CSV")
            output_path.write_bytes(content)
            status = "downloaded"

        download_log.append({"resource_number": resource_number,
                             "file_name": output_path.name, "status": status})

    display(pd.DataFrame(download_log))

    inspection = []
    for resource_number in sorted(valid_resources_df["resource_number"].astype(int)):
        file_path = RAW_TRAFFIC_DIR / f"traffic_resource_{resource_number}.csv"
        enc, sep = detect_csv(file_path)
        if enc is None:
            raise ValueError(f"Could not parse Resource {resource_number}")
        inspection.append({"resource_number": resource_number,
                           "encoding": enc, "separator": sep})

    traffic_file_inspection_df = pd.DataFrame(inspection)
    combined_parts = []
    for _, row in valid_resources_df.sort_values("resource_number").iterrows():
        resource_number = int(row["resource_number"])
        info = traffic_file_inspection_df[
            traffic_file_inspection_df["resource_number"] == resource_number
        ].iloc[0]
        file_path = RAW_TRAFFIC_DIR / f"traffic_resource_{resource_number}.csv"
        df = pd.read_csv(file_path, encoding=info["encoding"], sep=info["separator"],
                         low_memory=False, dtype=str, keep_default_na=False)
        df.insert(0, "resource_number", resource_number)
        df.insert(1, "source_file", file_path.name)
        year_match = re.search(r"20\d{2}", str(row["access_url"]))
        df.insert(2, "traffic_year", int(year_match.group()) if year_match else pd.NA)
        combined_parts.append(df)

    traffic_raw_combined = pd.concat(combined_parts, ignore_index=True, sort=False)
    traffic_raw_combined.to_csv(COMBINED_TRAFFIC_FILE, index=False, encoding="utf-8")
    traffic_raw_combined.to_parquet(TRAFFIC_PARQUET_FILE, index=False, engine="pyarrow")

    print("Traffic Parquet created:", TRAFFIC_PARQUET_FILE)
    print("Rows:", f"{len(traffic_raw_combined):,}")
    print("Columns:", len(traffic_raw_combined.columns))
    print("Resources:", traffic_raw_combined["resource_number"].nunique())
    print("Years:", sorted(traffic_raw_combined["traffic_year"].dropna().unique().tolist()))
    print("Raw number formatting preserved as text: PASS")


,resource_number,file_name,status
0,1,traffic_resource_1.csv,downloaded
1,2,traffic_resource_2.csv,downloaded
2,3,traffic_resource_3.csv,downloaded
3,4,traffic_resource_4.csv,downloaded
4,5,traffic_resource_5.csv,downloaded
5,6,traffic_resource_6.csv,downloaded
6,7,traffic_resource_7.csv,downloaded
7,8,traffic_resource_8.csv,downloaded
8,9,traffic_resource_9.csv,downloaded
9,10,traffic_resource_10.csv,downloaded


Traffic Parquet created: C:\Datenanalyse\final Project\Dataset_PlanA-B\Traffic\traffic_raw_combined.parquet
Rows: 36,577
Columns: 258
Resources: 22
Years: [2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Raw number formatting preserved as text: PASS


In [4]:
RAW_TABLE = "raw_bridge_statistics.traffic_raw"

expected_resources = set(
    valid_resources_df["resource_number"].astype(int)
)
actual_resources = set(
    traffic_raw_combined["resource_number"].astype(int).unique()
)

assert expected_resources == actual_resources, (
    f"Resource coverage mismatch: expected {sorted(expected_resources)}, "
    f"actual {sorted(actual_resources)}"
)

# Rebuild/synchronize PostgreSQL from the canonical in-memory Traffic dataset.
# The canonical dataset is loaded from Parquet on normal runs.
with engine.begin() as conn:
    conn.execute(
        text("CREATE SCHEMA IF NOT EXISTS raw_bridge_statistics")
    )
    conn.execute(
        text(f"DROP TABLE IF EXISTS {RAW_TABLE}")
    )

traffic_raw_combined.to_sql(
    "traffic_raw",
    engine,
    schema="raw_bridge_statistics",
    if_exists="replace",
    index=False,
    method="multi"
)

print("Raw PostgreSQL table rebuilt:", RAW_TABLE)
print("Raw number formatting preserved from CSV: PASS")
print("Resource coverage PASS:", expected_resources == actual_resources)
print("Combined rows:", f"{len(traffic_raw_combined):,}")
print("Full-row duplicates:", int(traffic_raw_combined.duplicated().sum()))


Raw PostgreSQL table rebuilt: raw_bridge_statistics.traffic_raw
Raw number formatting preserved from CSV: PASS
Resource coverage PASS: True
Combined rows: 36,577
Full-row duplicates: 0


## 3. Data Proofing & Cleaning

Audit row/resource coverage and identify constant technical columns. The raw table is not modified; cleaning is performed in a separate layer.

In [5]:
RAW_TABLE = "raw_bridge_statistics.traffic_raw"

raw_counts = pd.read_sql(
    text(f"SELECT COUNT(*) AS rows, COUNT(DISTINCT \"resource_number\") AS resources FROM {RAW_TABLE};"),
    engine
)
columns_df = pd.read_sql(
    text(f"""
        SELECT ordinal_position, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema='raw_bridge_statistics' AND table_name='traffic_raw'
        ORDER BY ordinal_position;
    """),
    engine
)

print("Rows:", int(raw_counts.loc[0, "rows"]))
print("Resources:", int(raw_counts.loc[0, "resources"]))
print("Columns:", len(columns_df))
print("Full-row duplicates:", int(traffic_raw_combined.duplicated().sum()))

constant_columns = []
for col in traffic_raw_combined.columns:
    if col in {"resource_number", "source_file", "traffic_year"}:
        continue
    if traffic_raw_combined[col].nunique(dropna=False) <= 1:
        constant_columns.append(col)

print("Constant columns to remove from cleaned layer:", len(constant_columns))
print(constant_columns)

Rows: 36577
Resources: 22
Columns: 258
Full-row duplicates: 0
Constant columns to remove from cleaned layer: 3
['TGMax2_SoFei_Ri1', 'TGMax2_SoFei_Ri2', 'Unnamed: 254']


In [6]:
CLEAN_TABLE = "cleaned.traffic_clean"
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS cleaned"))

keep_raw_columns = [c for c in traffic_raw_combined.columns if c not in constant_columns]
quoted = ", ".join('"' + c.replace('"', '""') + '"' for c in keep_raw_columns)
with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {CLEAN_TABLE}"))
    conn.execute(text(f"CREATE TABLE {CLEAN_TABLE} AS SELECT {quoted} FROM {RAW_TABLE}"))
print("Cleaned table created:", CLEAN_TABLE)

Cleaned table created: cleaned.traffic_clean


## 4. Traffic Core & Spatial Station Layer

Prepare the stable traffic core and create WGS84 point geometry from the traffic-station coordinates.

## 4. Traffic Core — Variable Selection

Keep identifiers, station/location information and traffic variables required for feature engineering.
All raw and cleaned data remain available; this is only a reduced working layer.

In [7]:
CLEAN_TABLE = "cleaned.traffic_clean"

# Key numeric fields
key_numeric_columns = [
    "Betriebs_km",
    "Koor_WGS84_N",
    "Koor_WGS84_E"
]

existing_clean_columns = pd.read_sql(
    text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema='cleaned'
          AND table_name='traffic_clean'
        ORDER BY ordinal_position
    """),
    engine
)
existing_names = set(existing_clean_columns["column_name"])

for column in key_numeric_columns:
    if column not in existing_names:
        continue

    dtype = existing_clean_columns.loc[
        existing_clean_columns["column_name"] == column,
        "data_type"
    ].iloc[0]

    if dtype in {"text", "character varying", "character"}:
        q = '"' + column.replace('"', '""') + '"'
        with engine.begin() as conn:
            conn.execute(text(f"""
                ALTER TABLE {CLEAN_TABLE}
                ALTER COLUMN {q} TYPE double precision
                USING NULLIF(
                    REPLACE(TRIM({q}), ',', '.'),
                    ''
                )::double precision
            """))

mandatory_columns = [
    "resource_number", "source_file", "traffic_year",
    "TK_Nr", "DZ_Nr", "DZ_Name",
    "Land_Nr", "Land_Code",
    "Str_Nr", "Str_Name",
    "Betriebs_km",
    "Koor_WGS84_N", "Koor_WGS84_E"
]

priority_groups = [
    ("DTV", 100, ["DTV"]),
    ("Heavy vehicles", 95, ["Lkw", "Lzg", "SV", "Schwer"]),
    ("Motor vehicles", 90, ["Kfz"]),
    ("Speed", 80, ["MSV", "Geschw", "Speed"]),
    ("Directional traffic", 70, ["Ri1", "Ri2"]),
]

profile=[]
for _, row in existing_clean_columns.iterrows():
    col=row["column_name"]
    if col in {"resource_number","source_file","traffic_year"}:
        continue

    series=(
        traffic_raw_combined[col]
        if col in traffic_raw_combined.columns
        else pd.Series(dtype=object)
    )
    missing_pct=float(series.isna().mean()*100) if len(series) else 100.0

    lower=col.lower()
    score=0
    category="Other"

    for group, base, tokens in priority_groups:
        if any(token.lower() in lower for token in tokens) and base > score:
            score=base
            category=group

    if col in mandatory_columns:
        score=1000
        category="Mandatory"

    profile.append({
        "column_name":col,
        "data_type":row["data_type"],
        "missing_pct":round(missing_pct,2),
        "category":category,
        "score":score
    })

column_selection_df=pd.DataFrame(profile)
column_selection_df["keep"]=(
    (column_selection_df["score"]>=70)
    & (column_selection_df["missing_pct"]<95)
)
column_selection_df.loc[
    column_selection_df["column_name"].isin(mandatory_columns),
    "keep"
]=True

# Retain every DTV-family column so semantic selection remains possible.
dtv_family_columns=[
    c for c in existing_names
    if "dtv" in c.lower()
]

selected_core_columns=[
    c for c in mandatory_columns if c in existing_names
]
selected_core_columns += [
    c for c in column_selection_df.loc[
        column_selection_df["keep"], "column_name"
    ]
    if c not in mandatory_columns
]
selected_core_columns += [
    c for c in dtv_family_columns
    if c not in selected_core_columns
]
selected_core_columns=list(dict.fromkeys(selected_core_columns))

print("Core columns selected:", len(selected_core_columns))
print("DTV-family columns retained:", dtv_family_columns)
display(column_selection_df.sort_values(
    ["keep","score","missing_pct"],
    ascending=[False,False,True]
))

Core columns selected: 183
DTV-family columns retained: ['DTV_Kfz_NZB_DiMiDo_Ri2', 'DTV_LoA_Ri1', 'DTV_Kfz_U_Ri2', 'DTV_Lzg_Ri2', 'DTV_SV_WU_MobisFr_Q', 'DTV_Kfz_U_Q', 'DTV_Lzg_Ri1', 'DTV_Kfz_U_Ri1', 'DTV_Kfz_WU_Sa_Q', 'DTV_SV_U_Q', 'DTV_Kfz_W_Ri2', 'DTV_Kfz_W_MobisFr_Q', 'DTV_Sat_Ri1', 'DTV_Kfz_NZB_DiMiDo_Ri1', 'DTV_SV_MobisSo_Ri2', 'DTV_Kfz_WU_Sa_Ri1', 'DTV_SV_WU_MobisFr_Ri2', 'DTV_SV_S_Q', 'DTV_SV_W_Ri1', 'DTV_Kfz_W_MobisFr_Ri1', 'DTV_LoA_Ri2', 'DTV_Kfz_S_Ri2', 'DTV_SV_S_Ri2', 'DTV_Kfz_WU_Sa_Ri2', 'DTV_Kfz_MobisSo_Q', 'DTV_Kfz_W_MobisFr_Ri2', 'DTV_Kfz_MobisSo_Ri1', 'DTV_SV_W_MobisFr_Ri2', 'DTV_SV_WU_Sa_Q', 'DTV_Kfz_W_Ri1', 'DTV_Bus_Ri1', 'DTV_Kfz_S_Ri1', 'DTV_Kfz_WU_MobisFr_Ri1', 'DTV_SV_W_MobisFr_Ri1', 'DTV_SV_W_MobisFr_Q', 'DTV_Bus_Ri2', 'DTV_SV_MobisSo_Ri1', 'DTV_Kfz_W_Q', 'DTV_SV_WU_MobisFr_Ri1', 'DTV_SV_S_Ri1', 'DTV_Kfz_MobisSo_Ri2', 'DTV_SV_MobisSo_Q', 'DTV_SV_U_Ri2', 'DTV_SV_W_Ri2', 'DTV_SV_WU_Sa_Ri1', 'DTV_Kfz_S_Q', 'DTV_SV_U_Ri1', 'DTV_Kfz_WU_MobisFr_Ri2', 'DTV_Sat_Ri2', 'D

,column_name,data_type,missing_pct,category,score,keep
0,TK_Nr,text,0.0,Mandatory,1000,True
1,DZ_Nr,text,0.0,Mandatory,1000,True
2,DZ_Name,text,0.0,Mandatory,1000,True
3,Land_Nr,text,0.0,Mandatory,1000,True
4,Land_Code,text,0.0,Mandatory,1000,True
...,...,...,...,...,...,...
227,Geo_Sequ,text,0.0,Other,0,False
228,Abschnitt_Ast,text,0.0,Other,0,False
229,Station,text,0.0,Other,0,False
230,Ausrichtung,text,0.0,Other,0,False


In [8]:
CORE_TABLE = "cleaned.traffic_core"

quoted_core=", ".join(
    '"' + c.replace('"','""') + '"'
    for c in selected_core_columns
)

with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {CORE_TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {CORE_TABLE} AS
        SELECT {quoted_core}
        FROM {CLEAN_TABLE}
    """))

print("Core traffic table created:", CORE_TABLE)
print("Columns:", len(selected_core_columns))

Core traffic table created: cleaned.traffic_core
Columns: 183


In [9]:
TRANSFORMED_TABLE = "transformed.traffic_station_geo"

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS transformed"))
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
    conn.execute(text(f"DROP TABLE IF EXISTS {TRANSFORMED_TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {TRANSFORMED_TABLE} AS
        SELECT *,
               CASE
                   WHEN "Koor_WGS84_E" BETWEEN -180 AND 180
                    AND "Koor_WGS84_N" BETWEEN -90 AND 90
                   THEN ST_SetSRID(
                       ST_MakePoint(
                           "Koor_WGS84_E",
                           "Koor_WGS84_N"
                       ),
                       4326
                   )
               END AS geom
        FROM {CORE_TABLE};
    """))
    conn.execute(
        text(
            f"CREATE INDEX traffic_station_geo_geom_idx "
            f"ON {TRANSFORMED_TABLE} USING GIST (geom);"
        )
    )

# Load the core table into pandas before performing dataframe-level checks.
core_df = pd.read_sql(
    text(f"SELECT * FROM {CORE_TABLE}"),
    engine
)

print("Spatial traffic table created:", TRANSFORMED_TABLE)
print("Core dataframe loaded:", core_df.shape)

core_missingness = (
    core_df.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("\nCore columns with highest missingness:")
display(
    core_missingness
    .head(15)
    .rename("missing_pct")
    .to_frame()
)

Spatial traffic table created: transformed.traffic_station_geo
Core dataframe loaded: (36577, 183)

Core columns with highest missingness:


,missing_pct
Betriebs_km,20.35
source_file,0.00
resource_number,0.00
TK_Nr,0.00
DZ_Nr,0.00
DZ_Name,0.00
traffic_year,0.00
Land_Nr,0.00
Land_Code,0.00
Str_Nr,0.00


## 5. Bridge Population — Strictly `Baujahr > 1900`

Use the Master Bridge table as the bridge population and apply the same strict rule used by the Bridge and Weather workflows.

In [10]:
BRIDGE_TABLE = "final.bridge"
MIN_BUILD_YEAR = 1900

# final.bridge is the canonical bridge population.
# final.bridge stores GIS coordinates as geom_x / geom_y rather than a PostGIS geom column.
bridge_population = pd.read_sql(
    text(f"""
        SELECT id_nr, bwnr, tbwnr, baujahr, geom_x, geom_y
        FROM {BRIDGE_TABLE}
        WHERE baujahr > :min_year
          AND baujahr IS NOT NULL
          AND geom_x IS NOT NULL
          AND geom_y IS NOT NULL
    """),
    engine,
    params={"min_year": MIN_BUILD_YEAR},
)

assert bridge_population["baujahr"].gt(MIN_BUILD_YEAR).all()
assert bridge_population["id_nr"].notna().all()
assert bridge_population["id_nr"].is_unique

print("Eligible bridges:", f"{len(bridge_population):,}")
print("Baujahr range:",
      int(bridge_population["baujahr"].min()),
      "-",
      int(bridge_population["baujahr"].max()))
print("PASS: only bridges with Baujahr > 1900 enter traffic mapping.")

Eligible bridges: 51,428
Baujahr range: 1901 - 3019
PASS: only bridges with Baujahr > 1900 enter traffic mapping.


## 6. Bridge → Nearest Traffic Station

For every eligible bridge, identify the nearest valid BASt traffic station with PostGIS.

**Acceptance rule:** `distance ≤ 5 km`. The calculated distance is retained as a quality-control feature.

In [11]:
from sqlalchemy import text
import pandas as pd

STATION_TABLE = "transformed.traffic_station_geo"
BRIDGE_TABLE = "final.bridge"
MAPPING_TABLE = "transformed.bridge_traffic_station"
MAX_TRAFFIC_MATCH_DISTANCE_KM = 5.0

station_schema = pd.read_sql(
    text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema='transformed'
          AND table_name='traffic_station_geo'
        ORDER BY ordinal_position
    """),
    engine
)
station_columns = set(station_schema["column_name"])

required_station_columns = [
    "TK_Nr", "DZ_Nr", "DZ_Name", "Betriebs_km",
    "traffic_year", "Koor_WGS84_N", "Koor_WGS84_E", "geom"
]
missing_required = [c for c in required_station_columns if c not in station_columns]
if missing_required:
    raise RuntimeError(
        "Required columns missing from traffic_station_geo: "
        + ", ".join(missing_required)
    )

def optional_column_sql(column_name, alias):
    if column_name in station_columns:
        return f's."{column_name}" AS {alias},'
    return f"NULL::text AS {alias},"

traffic_road_number_sql = optional_column_sql("Str_Nr", "traffic_road_number")
traffic_road_name_sql = optional_column_sql("Str_Name", "traffic_road_name")

with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {MAPPING_TABLE}"))

    conn.execute(text(f"""
        CREATE TABLE {MAPPING_TABLE} AS
        WITH bridge_geo AS (
            SELECT
                id_nr,
                bauwerk,
                bwnr,
                baujahr,
                geom_x,
                geom_y,
                CASE
                    -- WGS84 longitude/latitude
                    WHEN geom_x BETWEEN -180 AND 180
                     AND geom_y BETWEEN -90 AND 90
                    THEN ST_SetSRID(
                        ST_MakePoint(geom_x, geom_y), 4326
                    )

                    -- Web Mercator coordinates
                    WHEN ABS(geom_x) > 180
                     AND ABS(geom_y) > 90
                    THEN ST_Transform(
                        ST_SetSRID(
                            ST_MakePoint(geom_x, geom_y), 3857
                        ),
                        4326
                    )
                END AS geom
            FROM {BRIDGE_TABLE}
            WHERE baujahr > {MIN_BUILD_YEAR}
              AND baujahr IS NOT NULL
              AND geom_x IS NOT NULL
              AND geom_y IS NOT NULL
        )
        SELECT
            b.id_nr AS bridge_id,
            b.bauwerk,
            b.bwnr,
            b.baujahr,
            ST_Y(b.geom) AS bridge_latitude,
            ST_X(b.geom) AS bridge_longitude,

            s."TK_Nr" AS traffic_station_id,
            s."DZ_Nr" AS traffic_district_id,
            s."DZ_Name" AS traffic_district_name,
            {traffic_road_number_sql}
            {traffic_road_name_sql}
            s."Betriebs_km" AS traffic_operating_km,
            s."traffic_year",
            s."Koor_WGS84_N" AS traffic_latitude,
            s."Koor_WGS84_E" AS traffic_longitude,

            ST_Distance(
                b.geom::geography,
                s.geom::geography
            ) / 1000.0 AS distance_km,

            CASE
                WHEN ST_DWithin(
                    b.geom::geography,
                    s.geom::geography,
                    {MAX_TRAFFIC_MATCH_DISTANCE_KM} * 1000.0
                )
                THEN TRUE
                ELSE FALSE
            END AS spatial_match

        FROM bridge_geo AS b

        CROSS JOIN LATERAL (
            SELECT *
            FROM {STATION_TABLE}
            WHERE geom IS NOT NULL
            ORDER BY geom <-> b.geom
            LIMIT 1
        ) AS s

        WHERE b.geom IS NOT NULL
    """))

    conn.execute(text(
        f"CREATE INDEX IF NOT EXISTS bridge_traffic_station_bridge_idx "
        f"ON {MAPPING_TABLE} (bridge_id)"
    ))
    conn.execute(text(
        f"CREATE INDEX IF NOT EXISTS bridge_traffic_station_station_idx "
        f"ON {MAPPING_TABLE} (traffic_station_id)"
    ))

mapping = pd.read_sql(
    text(f"SELECT * FROM {MAPPING_TABLE} ORDER BY bridge_id"),
    engine
)

print("Bridge mappings:", f"{len(mapping):,}")
print("Unique bridges:", f"{mapping.bridge_id.nunique():,}")
print("PASS: nearest-station mapping created from final.bridge.")


Bridge mappings: 51,428
Unique bridges: 51,428
PASS: nearest-station mapping created from final.bridge.


## 7. Spatial Distance & Mapping Quality Control

Check uniqueness, distance distribution and the 5 km acceptance threshold before traffic features are transferred to bridges.

In [12]:
mapping_quality = pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS bridges_checked,
            COUNT(*) FILTER (WHERE spatial_match) AS matched_bridges,
            COUNT(*) FILTER (WHERE NOT spatial_match) AS unmatched_bridges,
            ROUND(
                100.0 * COUNT(*) FILTER (WHERE spatial_match)
                / NULLIF(COUNT(*),0), 2
            ) AS match_rate_pct,
            MIN(distance_km) AS min_distance_km,
            PERCENTILE_CONT(0.50)
                WITHIN GROUP (ORDER BY distance_km) AS median_distance_km,
            PERCENTILE_CONT(0.90)
                WITHIN GROUP (ORDER BY distance_km) AS p90_distance_km,
            MAX(distance_km) AS max_distance_km
        FROM {MAPPING_TABLE}
    """),
    engine
)
display(mapping_quality)

distance_distribution = pd.read_sql(
    text(f"""
        SELECT
            CASE
                WHEN distance_km <= 0.5 THEN '0-0.5 km'
                WHEN distance_km <= 1.0 THEN '0.5-1 km'
                WHEN distance_km <= 2.0 THEN '1-2 km'
                WHEN distance_km <= 5.0 THEN '2-5 km'
                ELSE '>5 km'
            END AS distance_band,
            COUNT(*) AS bridges
        FROM {MAPPING_TABLE}
        GROUP BY 1
        ORDER BY MIN(distance_km)
    """),
    engine
)
display(distance_distribution)

assert mapping["bridge_id"].is_unique
assert mapping["distance_km"].ge(0).all()
assert mapping["baujahr"].gt(MIN_BUILD_YEAR).all()

print("Accepted matches (<= 5 km):", int(mapping["spatial_match"].sum()))
print("Outside 5 km:", int((~mapping["spatial_match"]).sum()))
print("PASS: spatial quality control completed.")

,bridges_checked,matched_bridges,unmatched_bridges,match_rate_pct,min_distance_km,median_distance_km,p90_distance_km,max_distance_km
0,51428,31290,20138,60.84,0.000184,3.730806,12.009396,33.680128


,distance_band,bridges
0,0-0.5 km,4175
1,0.5-1 km,4391
2,1-2 km,7322
3,2-5 km,15402
4,>5 km,20138


Accepted matches (<= 5 km): 31290
Outside 5 km: 20138
PASS: spatial quality control completed.


## 8. Traffic Feature Selection

Use the actual traffic-core schema. Total DTV and heavy-vehicle traffic are separate variables.

In [13]:
# Discover the traffic variables from the actual traffic_core schema.
#
# DTV is an integer vehicle count per day (Kfz/d). The source files may use
# a period as a thousands separator, e.g. "2.312" means 2,312 vehicles/day.
# Raw values are therefore preserved as text until this field is normalized.

core_columns = pd.read_sql(
    text("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'cleaned'
          AND table_name = 'traffic_core'
        ORDER BY ordinal_position;
    """),
    engine
)

all_core_cols = core_columns["column_name"].tolist()

def qcol(c):
    return '"' + c.replace('"', '""') + '"'

TOTAL_DTV_CANDIDATES = [
    "DTV_Kfz_MobisSo_Q",
    "DTV_Kfz_MobisSo_Ri1",
    "DTV_Kfz_MobisSo_Ri2",
]

HEAVY_DTV_CANDIDATES = [
    "DTV_SV_MobisSo_Q",
    "DTV_SV_MobisSo_Ri1",
    "DTV_SV_MobisSo_Ri2",
]

dtv_col = next((c for c in TOTAL_DTV_CANDIDATES if c in all_core_cols), None)
heavy_col = next((c for c in HEAVY_DTV_CANDIDATES if c in all_core_cols), None)

print("Selected TOTAL-DTV column:", dtv_col)
print("Selected heavy-vehicle column:", heavy_col)

if dtv_col is None:
    raise RuntimeError(
        "The expected total-DTV field DTV_Kfz_MobisSo_Q was not found "
        "in cleaned.traffic_core."
    )

# BASt DTV is a vehicle count per day. Remove thousands separators from the
# original text representation:
#   "2.312" -> 2312
#   "12.345" -> 12345
#   "2312"  -> 2312
# This is intentionally applied only to the selected DTV field.
dtv_raw = qcol(dtv_col)
dtv_expr = f"""
CASE
    WHEN NULLIF(TRIM(CAST({dtv_raw} AS text)), '') IS NULL THEN NULL
    ELSE NULLIF(
        REPLACE(
            REPLACE(
                REPLACE(TRIM(CAST({dtv_raw} AS text)), '.', ''),
                ',', ''
            ),
            ' ', ''
        ),
        ''
    )::double precision
END
"""

print("DTV parsing rule: separators removed; result is vehicles/day (Kfz/d).")


Selected TOTAL-DTV column: DTV_Kfz_MobisSo_Q
Selected heavy-vehicle column: DTV_SV_MobisSo_Q
DTV parsing rule: separators removed; result is vehicles/day (Kfz/d).


## 9. Station-Year Traffic Features

Transform annual station observations into DTV, heavy vehicles, heavy-vehicle share, year-over-year growth and annual change.

**Unit handling:** DTV is parsed from the original BASt text representation as vehicle counts per day (Kfz/d), including thousands separators. Heavy-vehicle share is calculated as `heavy_vehicle_volume / dtv` using the same vehicle-count unit.


In [14]:
# Build station-year traffic features.
# DTV is normalized from the preserved raw text; the heavy-vehicle field is
# numeric-cast as stored in the source layer.

heavy_expr = (
    'NULL::double precision'
    if heavy_col is None
    else 'NULLIF(TRIM(CAST("'
         + heavy_col.replace('"', '""')
         + '" AS text)), \'\')::double precision'
)

with engine.begin() as conn:
    conn.execute(text(
        "DROP TABLE IF EXISTS transformed.traffic_station_year_features"
    ))
    conn.execute(text(f"""
        CREATE TABLE transformed.traffic_station_year_features AS
        WITH base AS (
            SELECT
                "TK_Nr" AS traffic_station_id,
                "traffic_year",
                NULLIF(
                    TRIM(CAST(({dtv_expr}) AS text)),
                    ''
                )::double precision AS dtv,
                {heavy_expr} AS heavy_vehicle_volume
            FROM cleaned.traffic_core
            WHERE "TK_Nr" IS NOT NULL
              AND "traffic_year" IS NOT NULL
        ),
        yearly AS (
            SELECT
                traffic_station_id,
                traffic_year,
                dtv,
                heavy_vehicle_volume,
                CASE
                    WHEN dtv > 0
                     AND heavy_vehicle_volume IS NOT NULL
                    THEN heavy_vehicle_volume / dtv
                    ELSE NULL
                END AS heavy_vehicle_share
            FROM base
        )
        SELECT
            *,
            dtv - LAG(dtv) OVER (
                PARTITION BY traffic_station_id
                ORDER BY traffic_year
            ) AS dtv_change,
            CASE
                WHEN LAG(dtv) OVER (
                    PARTITION BY traffic_station_id
                    ORDER BY traffic_year
                ) > 0
                THEN dtv / LAG(dtv) OVER (
                    PARTITION BY traffic_station_id
                    ORDER BY traffic_year
                ) - 1.0
                ELSE NULL
            END AS dtv_yoy_growth
        FROM yearly;
    """))

print("Created: transformed.traffic_station_year_features")


Created: transformed.traffic_station_year_features


In [15]:
# Validate the unit-consistent heavy-vehicle share.
share_check = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS valid_records,
            MIN(dtv) AS dtv_min,
            MAX(dtv) AS dtv_max,
            MIN(heavy_vehicle_share) AS min_share,
            MAX(heavy_vehicle_share) AS max_share,
            AVG(heavy_vehicle_share) AS mean_share,
            COUNT(*) FILTER (WHERE heavy_vehicle_share > 1) AS above_1,
            COUNT(*) FILTER (WHERE heavy_vehicle_share < 0) AS below_0
        FROM transformed.traffic_station_year_features
        WHERE heavy_vehicle_share IS NOT NULL
    """),
    engine
)

display(share_check)

row = share_check.iloc[0]

assert float(row["dtv_min"]) > 0, "QC failed: DTV must be > 0."
assert int(row["above_1"]) == 0, "QC failed: heavy_vehicle_share > 1"
assert int(row["below_0"]) == 0, "QC failed: heavy_vehicle_share < 0"

print("PASS: DTV is represented in vehicles/day (Kfz/d).")
print("PASS: heavy_vehicle_share is within [0, 1].")


,valid_records,dtv_min,dtv_max,min_share,max_share,mean_share,above_1,below_0
0,30476,222.0,184960.0,0.0,0.291955,0.024791,0,0


PASS: DTV is represented in vehicles/day (Kfz/d).
PASS: heavy_vehicle_share is within [0, 1].


## 10. Station-Level Traffic Features

Aggregate annual station time series into stable station-level summaries, including latest values, distribution statistics, growth and long-term trend.

In [16]:
# Aggregate the time series into one traffic feature record per station.
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS transformed.traffic_station_features"))
    conn.execute(text("""
        CREATE TABLE transformed.traffic_station_features AS
        SELECT
            traffic_station_id,
            MIN(traffic_year) FILTER (WHERE dtv IS NOT NULL) AS traffic_first_year,
            MAX(traffic_year) FILTER (WHERE dtv IS NOT NULL) AS traffic_last_year,
            COUNT(*) FILTER (WHERE dtv IS NOT NULL) AS traffic_years_available,
            COUNT(*) AS traffic_years_observed,

            MAX(dtv) AS dtv_max,
            MIN(dtv) AS dtv_min,
            AVG(dtv) AS dtv_mean,
            STDDEV_SAMP(dtv) AS dtv_std,

            MAX(dtv) FILTER (WHERE traffic_year = (
                SELECT MAX(t2.traffic_year)
                FROM transformed.traffic_station_year_features t2
                WHERE t2.traffic_station_id = t.traffic_station_id
                  AND t2.dtv IS NOT NULL
            )) AS dtv_latest,

            AVG(heavy_vehicle_volume) AS heavy_vehicle_mean,
            MAX(heavy_vehicle_volume) AS heavy_vehicle_max,
            AVG(heavy_vehicle_share) AS heavy_vehicle_share_mean,

            AVG(dtv_yoy_growth) AS dtv_yoy_growth_mean,
            MAX(dtv_yoy_growth) AS dtv_yoy_growth_max,
            MIN(dtv_yoy_growth) AS dtv_yoy_growth_min,

            CASE
                WHEN COUNT(dtv) >= 2
                THEN REGR_SLOPE(dtv, traffic_year)
                ELSE NULL
            END AS dtv_trend_per_year

        FROM transformed.traffic_station_year_features AS t
        GROUP BY traffic_station_id;
    """))

print("Created: transformed.traffic_station_features")

Created: transformed.traffic_station_features


## 11. Bridge-Level Traffic Features

Transfer only accepted (`≤5 km`) station features to the eligible bridge population.

In [17]:
BRIDGE_TRAFFIC_FEATURE_TABLE = "transformed.bridge_traffic_features"
FINAL_TRAFFIC_TABLE = "final.traffic"

with engine.begin() as conn:
    conn.execute(text(f"DROP TABLE IF EXISTS {BRIDGE_TRAFFIC_FEATURE_TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {BRIDGE_TRAFFIC_FEATURE_TABLE} AS
        SELECT
            m.bridge_id,
            m.traffic_station_id,
            m.distance_km AS traffic_station_distance_km,
            m.spatial_match,

            f.traffic_first_year,
            f.traffic_last_year,
            f.traffic_years_available,
            f.traffic_years_observed,
            f.dtv_latest,
            f.dtv_mean,
            f.dtv_max,
            f.dtv_min,
            f.dtv_std,
            f.heavy_vehicle_mean,
            f.heavy_vehicle_max,
            f.heavy_vehicle_share_mean,
            f.dtv_yoy_growth_mean,
            f.dtv_yoy_growth_max,
            f.dtv_yoy_growth_min,
            f.dtv_trend_per_year,

            CASE
                WHEN f.traffic_years_observed > 0
                THEN f.traffic_years_available::double precision
                     / f.traffic_years_observed
                ELSE NULL
            END AS traffic_data_coverage

        FROM {MAPPING_TABLE} AS m
        INNER JOIN transformed.traffic_station_features AS f
            ON f.traffic_station_id = m.traffic_station_id
        WHERE m.spatial_match = TRUE
          AND m.baujahr > {MIN_BUILD_YEAR}
    """))

    conn.execute(text(
        f"CREATE INDEX IF NOT EXISTS bridge_traffic_features_bridge_idx "
        f"ON {BRIDGE_TRAFFIC_FEATURE_TABLE} (bridge_id)"
    ))

print(f"Created: {BRIDGE_TRAFFIC_FEATURE_TABLE}")

final_quality = pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS bridge_traffic_records,
            COUNT(DISTINCT bridge_id) AS bridges_with_traffic,
            COUNT(DISTINCT traffic_station_id) AS traffic_stations_used,
            AVG(traffic_station_distance_km) AS mean_station_distance_km,
            MAX(traffic_station_distance_km) AS max_station_distance_km,
            AVG(traffic_data_coverage) AS mean_traffic_data_coverage
        FROM {BRIDGE_TRAFFIC_FEATURE_TABLE}
    """),
    engine
)
display(final_quality)

Created: transformed.bridge_traffic_features


,bridge_traffic_records,bridges_with_traffic,traffic_stations_used,mean_station_distance_km,max_station_distance_km,mean_traffic_data_coverage
0,31290,31290,1128,2.144438,4.999911,0.843446


## 12. Final Validation

Validate the bridge population rule, mapping threshold and the complete traffic feature set.

## 13. Promote Traffic Features to `final.traffic`

Promote the validated bridge-level traffic feature table to the canonical `final` schema for the downstream ML dataset.

**Source:** `transformed.bridge_traffic_features`  
**Final:** `final.traffic`

The transformed source remains available for reproducibility.

In [18]:
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS final"))
    conn.execute(text(f"DROP TABLE IF EXISTS {FINAL_TRAFFIC_TABLE}"))
    conn.execute(text(f"""
        CREATE TABLE {FINAL_TRAFFIC_TABLE} AS
        SELECT *
        FROM {BRIDGE_TRAFFIC_FEATURE_TABLE}
        WHERE spatial_match = TRUE
          AND bridge_id IS NOT NULL
    """))
    conn.execute(text(f"""
        CREATE UNIQUE INDEX idx_final_traffic_bridge
        ON {FINAL_TRAFFIC_TABLE}(bridge_id)
    """))

final_traffic_check = pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS bridge_traffic_records,
            COUNT(DISTINCT bridge_id) AS bridges_with_traffic,
            COUNT(DISTINCT traffic_station_id) AS traffic_stations_used,
            MIN(traffic_station_distance_km) AS min_station_distance_km,
            MAX(traffic_station_distance_km) AS max_station_distance_km
        FROM {FINAL_TRAFFIC_TABLE}
    """),
    engine
)
display(final_traffic_check)

ft = final_traffic_check.iloc[0]
assert int(ft["bridge_traffic_records"]) > 0
assert int(ft["bridges_with_traffic"]) == int(ft["bridge_traffic_records"])
assert float(ft["max_station_distance_km"]) <= MAX_TRAFFIC_MATCH_DISTANCE_KM + 1e-9

print(f"Final traffic table created: {FINAL_TRAFFIC_TABLE}")
print(f"Bridge records: {int(ft['bridge_traffic_records']):,}")
print(f"Traffic stations used: {int(ft['traffic_stations_used']):,}")
print("PASS: final.traffic is ready for downstream ML merging.")

,bridge_traffic_records,bridges_with_traffic,traffic_stations_used,min_station_distance_km,max_station_distance_km
0,31290,31290,1128,0.000184,4.999911


Final traffic table created: final.traffic
Bridge records: 31,290
Traffic stations used: 1,128
PASS: final.traffic is ready for downstream ML merging.


In [19]:
bridge_rule_check = pd.read_sql(
    text(f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(*) FILTER (
                WHERE b.baujahr <= {MIN_BUILD_YEAR}
                   OR b.baujahr IS NULL
            ) AS invalid_baujahr,
            COUNT(*) FILTER (
                WHERE f.traffic_station_distance_km > 5.0
            ) AS over_5km
        FROM final.traffic f
        JOIN {BRIDGE_TABLE} b
          ON b.id_nr = f.bridge_id
    """),
    engine
)
display(bridge_rule_check)

assert int(bridge_rule_check.loc[0, "invalid_baujahr"]) == 0
assert int(bridge_rule_check.loc[0, "over_5km"]) == 0

feature_columns = pd.read_sql(
    text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema='transformed'
          AND table_name='bridge_traffic_features'
        ORDER BY ordinal_position
    """),
    engine
)["column_name"].tolist()

required_features = [
    "dtv_latest","dtv_mean","dtv_max","dtv_min","dtv_std",
    "heavy_vehicle_mean","heavy_vehicle_max","heavy_vehicle_share_mean",
    "dtv_yoy_growth_mean","dtv_yoy_growth_max","dtv_yoy_growth_min",
    "dtv_trend_per_year","traffic_data_coverage"
]
missing_features=[c for c in required_features if c not in feature_columns]
assert not missing_features, f"Missing traffic features: {missing_features}"

print("PASS: Baujahr > 1900, <=5 km and all required traffic features validated.")

,rows,invalid_baujahr,over_5km
0,31290,0,0


PASS: Baujahr > 1900, <=5 km and all required traffic features validated.


## 14. Final Results / Brief

In [20]:
result = pd.read_sql(
    text("""
        SELECT
            COUNT(*) AS bridge_traffic_records,
            COUNT(DISTINCT bridge_id) AS bridges_with_traffic,
            COUNT(DISTINCT traffic_station_id) AS traffic_stations_used,
            ROUND(AVG(traffic_station_distance_km)::numeric, 3)
                AS mean_station_distance_km,
            ROUND(PERCENTILE_CONT(0.50)
                WITHIN GROUP (ORDER BY traffic_station_distance_km)::numeric, 3)
                AS median_station_distance_km,
            ROUND(MAX(traffic_station_distance_km)::numeric, 3)
                AS max_station_distance_km,
            ROUND(AVG(traffic_data_coverage)::numeric, 3)
                AS mean_traffic_data_coverage
        FROM transformed.bridge_traffic_features
        WHERE spatial_match = TRUE
    """),
    engine
)
display(result)

r=result.iloc[0]
print("="*70)
print("FINAL TRAFFIC FEATURE BRIEF")
print("="*70)
print(f"Bridges with traffic features: {int(r.bridge_traffic_records):,}")
print(f"Traffic stations used: {int(r.traffic_stations_used):,}")
print(f"Mean station distance: {r.mean_station_distance_km} km")
print(f"Median station distance: {r.median_station_distance_km} km")
print(f"Maximum accepted distance: {r.max_station_distance_km} km")
print(f"Mean traffic data coverage: {r.mean_traffic_data_coverage}")
print("\nOutput: final.traffic")

,bridge_traffic_records,bridges_with_traffic,traffic_stations_used,mean_station_distance_km,median_station_distance_km,max_station_distance_km,mean_traffic_data_coverage
0,31290,31290,1128,2.144,1.965,5.0,0.843


FINAL TRAFFIC FEATURE BRIEF
Bridges with traffic features: 31,290
Traffic stations used: 1,128
Mean station distance: 2.144 km
Median station distance: 1.965 km
Maximum accepted distance: 5.0 km
Mean traffic data coverage: 0.843

Output: final.traffic


# Final Status

### Output
`final.traffic` (source retained as `transformed.bridge_traffic_features`).

### QC note
DTV is parsed from the original BASt text representation as vehicles/day, including thousands separators. `heavy_vehicle_share` is calculated as `heavy_vehicle_volume / dtv` using the same unit.
